In [2]:

from collections import deque
import heapq
import time


def take_maze():

    r = int(input("Rows: "))
    c = int(input("Columns: "))

    maze = []

    print("Enter 0 for path and 1 for wall")

    for i in range(r):
        while True:
            row = input("Row " + str(i + 1) + ": ").replace(" ", "")

            if len(row) == c and all(x in "01" for x in row):
                maze.append([int(x) for x in row])
                break

            print("Enter", c, "digits using only 0 and 1")

    print("Enter start position")

    sr = int(input("Start row: "))
    sc = int(input("Start column: "))

    print("Enter goal position")

    gr = int(input("Goal row: "))
    gc = int(input("Goal column: "))

    start = (sr, sc)
    goal = (gr, gc)

    if not (0 <= sr < r and 0 <= sc < c):
        raise ValueError("Invalid start position")

    if not (0 <= gr < r and 0 <= gc < c):
        raise ValueError("Invalid goal position")

    if maze[sr][sc] == 1 or maze[gr][gc] == 1:
        raise ValueError("Start and goal must be free cells")

    return maze, start, goal


def neighbours(pos, maze):

    row, col = pos

    moves = [
        (-1, 0),
        (1, 0),
        (0, -1),
        (0, 1)
    ]

    result = []

    for dr, dc in moves:

        nr = row + dr
        nc = col + dc

        if 0 <= nr < len(maze) and 0 <= nc < len(maze[0]):

            if maze[nr][nc] == 0:
                result.append((nr, nc))

    return result


def make_path(parent, start, goal):

    if goal not in parent:
        return []

    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()

    return path


def bfs(maze, start, goal):

    start_time = time.perf_counter()

    queue = deque([start])
    visited = {start}
    parent = {start: None}

    generated = 1
    expanded = 0
    max_frontier = 1

    while queue:

        current = queue.popleft()
        expanded += 1

        if current == goal:
            break

        for next_pos in neighbours(current, maze):

            if next_pos not in visited:

                visited.add(next_pos)
                parent[next_pos] = current
                queue.append(next_pos)
                generated += 1

        max_frontier = max(max_frontier, len(queue))

    path = make_path(parent, start, goal)

    elapsed = (time.perf_counter() - start_time) * 1000

    return {
        "Algorithm": "BFS",
        "Path": path,
        "Path Length": len(path) - 1 if path else "No path",
        "Generated": generated,
        "Expanded": expanded,
        "Frontier": max_frontier,
        "Time": round(elapsed, 4)
    }


def dfs(maze, start, goal):

    start_time = time.perf_counter()

    stack = [start]
    visited = {start}
    parent = {start: None}

    generated = 1
    expanded = 0
    max_frontier = 1

    while stack:

        current = stack.pop()
        expanded += 1

        if current == goal:
            break

        for next_pos in neighbours(current, maze):

            if next_pos not in visited:

                visited.add(next_pos)
                parent[next_pos] = current
                stack.append(next_pos)
                generated += 1

        max_frontier = max(max_frontier, len(stack))

    path = make_path(parent, start, goal)

    elapsed = (time.perf_counter() - start_time) * 1000

    return {
        "Algorithm": "DFS",
        "Path": path,
        "Path Length": len(path) - 1 if path else "No path",
        "Generated": generated,
        "Expanded": expanded,
        "Frontier": max_frontier,
        "Time": round(elapsed, 4)
    }


def heuristic(pos, goal):

    return abs(pos[0] - goal[0]) + abs(pos[1] - goal[1])


def astar(maze, start, goal):

    start_time = time.perf_counter()

    queue = []
    count = 0

    heapq.heappush(
        queue,
        (heuristic(start, goal), count, start)
    )

    parent = {start: None}
    cost = {start: 0}
    visited = set()

    generated = 1
    expanded = 0
    max_frontier = 1

    while queue:

        f, old_count, current = heapq.heappop(queue)

        if current in visited:
            continue

        visited.add(current)
        expanded += 1

        if current == goal:
            break

        for next_pos in neighbours(current, maze):

            if next_pos in visited:
                continue

            new_cost = cost[current] + 1

            if next_pos not in cost or new_cost < cost[next_pos]:

                cost[next_pos] = new_cost
                parent[next_pos] = current

                h = heuristic(next_pos, goal)
                f = new_cost + h

                count += 1

                heapq.heappush(
                    queue,
                    (f, count, next_pos)
                )

                generated += 1

        max_frontier = max(max_frontier, len(queue))

    path = make_path(parent, start, goal)

    elapsed = (time.perf_counter() - start_time) * 1000

    return {
        "Algorithm": "A*",
        "Path": path,
        "Path Length": len(path) - 1 if path else "No path",
        "Generated": generated,
        "Expanded": expanded,
        "Frontier": max_frontier,
        "Time": round(elapsed, 4)
    }


def show_result(result):

    print("\n", result["Algorithm"])

    print("Path:", result["Path"])
    print("Path length:", result["Path Length"])
    print("Nodes generated:", result["Generated"])
    print("Nodes expanded:", result["Expanded"])
    print("Maximum frontier:", result["Frontier"])
    print("Time:", result["Time"], "ms")


def show_table(results):

    print("\nComparison")

    print(
        "{:<12} {:<12} {:<12} {:<12} {:<12} {:<12}".format(
            "Algorithm",
            "Path",
            "Generated",
            "Expanded",
            "Frontier",
            "Time"
        )
    )

    for result in results:

        print(
            "{:<12} {:<12} {:<12} {:<12} {:<12} {:<12}".format(
                result["Algorithm"],
                result["Path Length"],
                result["Generated"],
                result["Expanded"],
                result["Frontier"],
                result["Time"]
            )
        )


def run_maze(number):

    print("\nMaze", number)

    maze, start, goal = take_maze()

    results = [
        bfs(maze, start, goal),
        dfs(maze, start, goal),
        astar(maze, start, goal)
    ]

    for result in results:
        show_result(result)

    show_table(results)

    return results


print("Maze Pathfinding using BFS, DFS and A*")

n = int(input("How many mazes: "))

all_results = []

for i in range(1, n + 1):

    results = run_maze(i)

    for result in results:

        all_results.append({
            "Maze": i,
            "Algorithm": result["Algorithm"],
            "Path Length": result["Path Length"],
            "Generated": result["Generated"],
            "Expanded": result["Expanded"],
            "Frontier": result["Frontier"],
            "Time": result["Time"]
        })


print("\nFinal results")

print(
    "{:<8} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12}".format(
        "Maze",
        "Algorithm",
        "Path",
        "Generated",
        "Expanded",
        "Frontier",
        "Time"
    )
)

for result in all_results:

    print(
        "{:<8} {:<12} {:<12} {:<12} {:<12} {:<12} {:<12}".format(
            result["Maze"],
            result["Algorithm"],
            result["Path Length"],
            result["Generated"],
            result["Expanded"],
            result["Frontier"],
            result["Time"]
        )
    )

Maze Pathfinding using BFS, DFS and A*


How many mazes:  3



Maze 1


Rows:  5
Columns:  5


Enter 0 for path and 1 for wall


Row 1:  00010
Row 2:  11010
Row 3:  00000
Row 4:  01110
Row 5:  00000


... (17 outputs left out)

Start row:  0
Start column:  0


Enter goal position


Goal row:  6
Goal column:  6



 BFS
Path: [(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (2, 3), (1, 3), (0, 3), (0, 4), (0, 5), (0, 6), (1, 6), (2, 6), (3, 6), (4, 6), (5, 6), (6, 6)]
Path length: 16
Nodes generated: 22
Nodes expanded: 21
Maximum frontier: 3
Time: 0.1 ms

 DFS
Path: [(0, 0), (0, 1), (1, 1), (2, 1), (2, 2), (2, 3), (1, 3), (0, 3), (0, 4), (0, 5), (0, 6), (1, 6), (2, 6), (3, 6), (4, 6), (5, 6), (6, 6)]
Path length: 16
Nodes generated: 25
Nodes expanded: 24
Maximum frontier: 4
Time: 0.0 ms

 A*
Path: [(0, 0), (1, 0), (1, 1), (2, 1), (2, 2), (2, 3), (1, 3), (0, 3), (0, 4), (0, 5), (0, 6), (1, 6), (2, 6), (3, 6), (4, 6), (5, 6), (6, 6)]
Path length: 16
Nodes generated: 22
Nodes expanded: 19
Maximum frontier: 4
Time: 1.1 ms

Comparison
Algorithm    Path         Generated    Expanded     Frontier     Time        
BFS          16           22           21           3            0.1         
DFS          16           25           24           4            0.0         
A*           16           22           19   